In [1]:
import numpy as np
import scipy as sp

In [6]:
g = 0.2       # quartic coefficient
m = 1       # mass
w = 1       # supposed to be omega i.e. frequency of harmonic oscillator
hb = 1      # short for hbar

b = 10.0    # integral upper bound for Gauss-Legendre quadrature
a = -10.0   # integral lower bound for Gauss-Legendre quadrature (G-L Q)

c1 = (b-a)/2    # these two helper variables are convenient to calculate change of interval for G-L Q
c2 = (b+a)/2

samples, weights = np.polynomial.legendre.leggauss(99)      # G-L Q sample points & weights

#the wavefunction that solves the unperturbed hamiltonian
def wavefunction(n, x, Omega):
    freq = np.sqrt(w**2 + Omega**2)
    xi = x * np.sqrt(m*freq/hb)
    return np.exp(-(xi**2)/2) * (m*freq /(np.pi*hb))**(1/4) * ((2**n)*sp.special.factorial(n))**(-1/2) * sp.special.eval_hermite(n, xi)

#perturbation to the hamiltonian
def H1(x, Omega):
    return g*(x**4) - (1.0/2)*m*(Omega**2)*(x**2)

#helper function to determine the integrand for calculating the expectation value
def trans_amp(n1, n2, x, Omega):
    return wavefunction(n1, x, Omega) * H1(x, Omega) * wavefunction(n2, x, Omega)

def E0(n, Omega):
    return (n*1.0 + 1.0/2)*hb*np.sqrt(w**2 + Omega**2)

#first order energy
def E1(n, Omega):
    return c1 * np.dot(weights, trans_amp(n, n, c1*samples + c2, Omega))

#second order energy
def E2(n, Omega):
    E_2 = 0
    # with some more analysis you can show that you can skip every other term, but we'll keep it general
    for i in range(0, n + 5):
        if (i == n):
            continue 

        transition_amplitude = c1 * np.dot(weights, trans_amp(n, i, c1*samples + c2, Omega))
        E_2 += transition_amplitude**2 / (E0(n, Omega) - E0(i, Omega))

    return E_2

def ground(Omega):
    return E0(0, Omega) + E1(0, Omega) + E2(0, Omega)

In [16]:
print(ground(-0.5))

0.5842147027303836


In [21]:
min_E = sp.optimize.minimize(ground, x0 = [1])

print(min_E)

  message: Optimization terminated successfully.
  success: True
   status: 0
      fun: 0.5449999999999999
        x: [-1.539e-08]
      nit: 3
      jac: [ 7.451e-09]
 hess_inv: [[ 1.905e+00]]
     nfev: 24
     njev: 12
